# MedTrack_DV — Module 2: Data Cleaning & Transformation

**Milestone 1, Week 1–2**

Goal: take `hospital_raw_data.csv` (Module 1 output) and produce a clean,
Tableau-ready dataset: `hospital_cleaned.csv`.

Tasks covered in this notebook:
1. Remove duplicate records
2. Handle missing patient data
3. Standardize department names
4. Normalize healthcare indicators
5. Create Tableau-ready dataset (correct types + a few analysis-friendly fields)

Even though Module 1's output was already fairly clean (it was integrated in
the same pipeline), this notebook is written to be **robust** — it will
correctly find and fix duplicates, missing values, and inconsistent text
regardless of how dirty the input actually is. That matters if you swap in a
different/real raw file later.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Load the raw dataset

In [2]:
df = pd.read_csv("hospital_raw_data.csv")
print(f"Rows: {len(df)}  |  Columns: {len(df.columns)}")
df.head()

Rows: 984  |  Columns: 16


,Patient_ID,Hospital,Region,Department,Patient_Type,Admission_Date,Discharge_Date,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
0,1,City Care Hospital,East,Cardiology,Inpatient,2024-07-10,2024-07-15,45,Female,Heart Disease,Angioplasty,15000,5,No,Recovered,4
1,2,Metro Health Institute,West,General Medicine,Emergency,2024-06-21,2024-06-24,60,Male,Diabetes,Insulin Therapy,2000,3,Yes,Stable,3
2,3,Metro Health Institute,West,Orthopedics,Inpatient,2024-09-21,2024-09-22,32,Female,Fractured Arm,X-Ray and Splint,500,1,No,Recovered,5
3,4,Sunrise Medical Center,South,ICU,Inpatient,2024-07-05,2024-07-12,75,Male,Stroke,CT Scan and Medication,10000,7,Yes,Stable,2
4,5,Sunrise Medical Center,South,Surgery,Inpatient,2024-07-18,2024-07-28,50,Female,Cancer,Surgery and Chemotherapy,25000,10,No,Recovered,4


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Patient_ID      984 non-null    int64
 1   Hospital        984 non-null    str  
 2   Region          984 non-null    str  
 3   Department      984 non-null    str  
 4   Patient_Type    984 non-null    str  
 5   Admission_Date  984 non-null    str  
 6   Discharge_Date  984 non-null    str  
 7   Age             984 non-null    int64
 8   Gender          984 non-null    str  
 9   Condition       984 non-null    str  
 10  Procedure       984 non-null    str  
 11  Cost            984 non-null    int64
 12  Length_of_Stay  984 non-null    int64
 13  Readmission     984 non-null    str  
 14  Outcome         984 non-null    str  
 15  Satisfaction    984 non-null    int64
dtypes: int64(5), str(11)
memory usage: 123.1 KB


## 2. Remove duplicate records

In [4]:
before = len(df)

# Exact duplicate rows
exact_dupes = df.duplicated().sum()

# Duplicate Patient_ID (a patient should only have one admission record here)
id_dupes = df.duplicated(subset=["Patient_ID"]).sum()

print(f"Exact duplicate rows: {exact_dupes}")
print(f"Duplicate Patient_IDs: {id_dupes}")

df = df.drop_duplicates()
df = df.drop_duplicates(subset=["Patient_ID"], keep="first")

after = len(df)
print(f"\nRows removed: {before - after}  |  Remaining rows: {after}")

Exact duplicate rows: 0
Duplicate Patient_IDs: 0

Rows removed: 0  |  Remaining rows: 984


## 3. Handle missing patient data

In [5]:
missing_before = df.isna().sum()
print("Missing values by column (before):")
print(missing_before[missing_before > 0] if missing_before.sum() else "None found")

Missing values by column (before):
None found


In [6]:
# Strategy per column type:
# - Categorical (Department, Hospital, Region, Gender, Condition, Procedure,
#   Patient_Type, Outcome, Readmission): fill with "Unknown"
# - Numeric (Age, Cost, Length_of_Stay, Satisfaction): fill with column median
# - Dates (Admission_Date, Discharge_Date): drop the row — a record with no
#   admission date can't be placed in any time-based KPI, so imputing it
#   would create misleading trends

categorical_cols = ["Hospital", "Region", "Department", "Patient_Type",
                     "Gender", "Condition", "Procedure", "Readmission", "Outcome"]
numeric_cols = ["Age", "Cost", "Length_of_Stay", "Satisfaction"]
date_cols = ["Admission_Date", "Discharge_Date"]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

rows_before_date_drop = len(df)
df = df.dropna(subset=[c for c in date_cols if c in df.columns])
print(f"Rows dropped for missing dates: {rows_before_date_drop - len(df)}")

missing_after = df.isna().sum()
completeness = 100 * (1 - missing_after.sum() / df.size)
print(f"\nRemaining missing values: {missing_after.sum()}")
print(f"Dataset completeness: {completeness:.2f}%")
assert missing_after.sum() / df.size < 0.02, "Missing values exceed 2% target"
print("Missing-values target (<2%) met ✔")

Rows dropped for missing dates: 0

Remaining missing values: 0
Dataset completeness: 100.00%
Missing-values target (<2%) met ✔


## 4. Standardize department names (and other text fields)

In [7]:
# Trim whitespace and fix casing so the same department/hospital/region
# isn't split into multiple categories in Tableau (e.g. "cardiology " vs "Cardiology")

VALID_DEPARTMENTS = {
    "cardiology": "Cardiology",
    "general medicine": "General Medicine",
    "orthopedics": "Orthopedics",
    "orthopaedics": "Orthopedics",
    "icu": "ICU",
    "surgery": "Surgery",
    "emergency": "Emergency",
    "pediatrics": "Pediatrics",
    "paediatrics": "Pediatrics",
}

def standardize_department(value):
    key = str(value).strip().lower()
    return VALID_DEPARTMENTS.get(key, str(value).strip().title())

df["Department"] = df["Department"].apply(standardize_department)

text_cols = ["Hospital", "Region", "Gender", "Condition", "Procedure", "Outcome"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Readmission as consistent Yes/No text
df["Readmission"] = df["Readmission"].astype(str).str.strip().str.title()
df["Readmission"] = df["Readmission"].replace({"1": "Yes", "0": "No", "True": "Yes", "False": "No"})

print("Departments after standardization:")
print(df["Department"].value_counts())

Departments after standardization:
Department
General Medicine    261
Orthopedics         197
Surgery             197
Cardiology          132
ICU                  66
Emergency            66
Pediatrics           65
Name: count, dtype: int64


## 5. Normalize healthcare indicators

In [8]:
# Correct data types
df["Admission_Date"] = pd.to_datetime(df["Admission_Date"], errors="coerce")
df["Discharge_Date"] = pd.to_datetime(df["Discharge_Date"], errors="coerce")

df["Age"] = pd.to_numeric(df["Age"], errors="coerce").astype("Int64")
df["Cost"] = pd.to_numeric(df["Cost"], errors="coerce").round(2)
df["Length_of_Stay"] = pd.to_numeric(df["Length_of_Stay"], errors="coerce").astype("Int64")
df["Satisfaction"] = pd.to_numeric(df["Satisfaction"], errors="coerce").astype("Int64")

# Sanity-check ranges and fix impossible values
before = len(df)
df = df[(df["Age"] >= 0) & (df["Age"] <= 120)]
df = df[df["Length_of_Stay"] >= 0]
df = df[df["Cost"] >= 0]
df = df[df["Satisfaction"].between(1, 5)]
df = df[df["Discharge_Date"] >= df["Admission_Date"]]
print(f"Rows dropped for out-of-range / inconsistent values: {before - len(df)}")

# Consistent boolean-style flag alongside the Yes/No text (useful for Tableau calcs)
df["Is_Readmitted"] = df["Readmission"].map({"Yes": 1, "No": 0}).fillna(0).astype(int)

Rows dropped for out-of-range / inconsistent values: 0


## 6. Create Tableau-ready dataset

Add a few lightweight derived fields Tableau dashboards will use directly (avoids repeating date-math in every calculated field).

In [9]:
df["Admission_Year"] = df["Admission_Date"].dt.year
df["Admission_Month"] = df["Admission_Date"].dt.to_period("M").astype(str)
df["Admission_Month_Name"] = df["Admission_Date"].dt.strftime("%b")

column_order = [
    "Patient_ID", "Hospital", "Region", "Department", "Patient_Type",
    "Admission_Date", "Discharge_Date", "Admission_Year", "Admission_Month",
    "Admission_Month_Name", "Age", "Gender", "Condition", "Procedure", "Cost",
    "Length_of_Stay", "Readmission", "Is_Readmitted", "Outcome", "Satisfaction",
]
df = df[column_order].sort_values("Admission_Date").reset_index(drop=True)
df.head()

,Patient_ID,Hospital,Region,Department,Patient_Type,Admission_Date,Discharge_Date,Admission_Year,Admission_Month,Admission_Month_Name,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Is_Readmitted,Outcome,Satisfaction
0,767,Sunrise Medical Center,North,General Medicine,Inpatient,2024-01-01,2024-02-24,2024,2024-01,Jan,62,Male,Diabetes,Insulin Therapy,2000,54,No,0,Stable,4
1,177,Metro Health Institute,South,Surgery,Inpatient,2024-01-02,2024-01-22,2024,2024-01,Jan,67,Male,Prostate Cancer,Radiation Therapy,20000,20,No,0,Recovered,3
2,673,Metro Health Institute,South,Pediatrics,Inpatient,2024-01-02,2024-02-18,2024,2024-01,Jan,32,Female,Childbirth,Delivery and Postnatal Care,12000,47,No,0,Recovered,4
3,702,HealthPlus Hospital,West,Surgery,Outpatient,2024-01-02,2024-02-26,2024,2024-01,Jan,58,Male,Prostate Cancer,Radiation Therapy,20000,55,No,0,Recovered,3
4,432,Sunrise Medical Center,North,Surgery,Outpatient,2024-01-03,2024-02-09,2024,2024-01,Jan,58,Male,Prostate Cancer,Radiation Therapy,20000,37,No,0,Recovered,3


## 7. Final validation

In [10]:
missing_pct = 100 * df.isna().sum().sum() / df.size
dupe_count = df.duplicated(subset=["Patient_ID"]).sum()

print(f"Final rows: {len(df)}")
print(f"Missing values: {missing_pct:.2f}%  (target < 2%)")
print(f"Duplicate Patient_IDs: {dupe_count}  (target: 0)")
print(f"Departments: {sorted(df['Department'].unique())}")
print(f"Date range: {df['Admission_Date'].min().date()} to {df['Admission_Date'].max().date()}")

assert missing_pct < 2, "Missing values exceed target"
assert dupe_count == 0, "Duplicate patient records remain"
print("\nAll evaluation checks passed ✔")

Final rows: 984
Missing values: 0.00%  (target < 2%)
Duplicate Patient_IDs: 0  (target: 0)
Departments: ['Cardiology', 'Emergency', 'General Medicine', 'ICU', 'Orthopedics', 'Pediatrics', 'Surgery']
Date range: 2024-01-01 to 2024-12-31

All evaluation checks passed ✔


## 8. Save cleaned dataset

In [11]:
df.to_csv("hospital_cleaned.csv", index=False)
print(f"Saved hospital_cleaned.csv  |  {len(df)} rows, {len(df.columns)} columns")

Saved hospital_cleaned.csv  |  984 rows, 20 columns


## Summary

| Item | Value |
|---|---|
| Input file | `hospital_raw_data.csv` |
| Output file | `hospital_cleaned.csv` |
| Duplicates removed | 0 (none present) |
| Missing values | 0.00% (target < 2%) |
| Departments standardized | 7 consistent categories |
| New fields added | `Admission_Year`, `Admission_Month`, `Admission_Month_Name`, `Is_Readmitted` |

**Next:** Module 3 — Hospital KPI Engineering (`generate_hospital_kpis.py`).